# Taxon AE Visualization

Clean visualization workflow for the modular Taxon ResNet autoencoder with model loading.

In [ ]:
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils

from taxon_ae import TaxonAutoencoder


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = TaxonAutoencoder(
    in_channels=3,
    resnet_variant='18',
    stage_taxonomy_layers=(5, 6, 7, 8),
    stage_strides=(1, 2, 2, 2),
    temperature=1.0,
    hard=False,
    use_stem=True,
    stem_channels=64,
    stem_stride=2,
    use_stem_maxpool=True,
    output_activation='none',
).to(device)

checkpoint_path = Path('./outputs/taxon_ae_celeba_hq/checkpoints/best.pt')
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint.get('model_state', checkpoint)
    model.load_state_dict(state_dict, strict=False)
    print(f'Loaded checkpoint: {checkpoint_path}')
else:
    print(f'Checkpoint not found: {checkpoint_path}. Using random initialization.')

model.eval()


In [ ]:
class CelebAHQImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = Path(root)
        self.transform = transform
        exts = {'.jpg', '.jpeg', '.png', '.webp'}
        self.files = sorted([p for p in self.root.rglob('*') if p.suffix.lower() in exts])
        if len(self.files) == 0:
            raise FileNotFoundError(f'No image files found under {self.root}')

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        image = Image.open(self.files[idx]).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, 0

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

dataset_root = Path('/nethome/zwang910/file_storage/datasets/CelebA-HQ')
dataset = CelebAHQImageDataset(dataset_root, transform=transform)
loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2)

images, _ = next(iter(loader))
images = images.to(device)


In [ ]:
@torch.no_grad()
def run_model_with_details(model, images, hard=False):
    recon, dkl, entropy, details = model(images, hard=hard, return_details=True)
    return recon, dkl, entropy, details

def show_input_reconstruction(images, recon, max_images=4):
    vis_in = (images[:max_images].detach().cpu().clamp(-1, 1) + 1.0) * 0.5
    vis_rec = (recon[:max_images].detach().cpu().clamp(-1, 1) + 1.0) * 0.5
    grid = utils.make_grid(torch.cat([vis_in, vis_rec], dim=0), nrow=max_images, padding=2)

    plt.figure(figsize=(3 * max_images, 6))
    plt.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
    plt.axis('off')
    plt.title('Top row: input, Bottom row: reconstruction')
    plt.show()

recon, dkl, entropy, details = run_model_with_details(model, images, hard=False)
print(f'dkl={float(dkl):.6f}, entropy={float(entropy):.6f}')
show_input_reconstruction(images, recon, max_images=4)


In [ ]:
def plot_patch_path_probabilities(stage_info, sample_index=0, max_patches=100, alpha=0.7):
    logp = stage_info['logp'][sample_index:sample_index+1]
    layer_channels = stage_info['layer_channels']

    prob_splits = [torch.exp(t) for t in torch.split(logp, layer_channels, dim=1)]
    flattened = torch.cat(prob_splits, dim=1)
    layer_sums_map = torch.stack([p.sum(dim=1) for p in prob_splits], dim=1)

    patch_probs = flattened[0].permute(1, 2, 0).reshape(-1, flattened.shape[1])
    layer_sums = layer_sums_map[0].permute(1, 2, 0).reshape(-1, len(layer_channels))

    num_patches = min(max_patches, patch_probs.shape[0])
    patch_probs = patch_probs[:num_patches]
    layer_sums = layer_sums[:num_patches]

    x_full = np.arange(patch_probs.shape[1])
    boundaries = np.cumsum(layer_channels)[:-1] - 0.5

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [2.4, 1.2]})

    ax = axes[0]
    start = 0
    layer_slices = []
    for width in layer_channels:
        end = start + width
        layer_slices.append((x_full[start:end], start, end))
        start = end

    for i in range(num_patches):
        y = patch_probs[i].detach().cpu().numpy()
        for x_seg, s, e in layer_slices:
            ax.plot(x_seg, y[s:e], alpha=alpha, linewidth=1.0)

    for b in boundaries:
        ax.axvline(b, color='gray', linestyle='--', linewidth=0.8)

    ax.set_xlabel('Flattened tree channel index')
    ax.set_ylabel('Probability')
    ax.set_title(f"{stage_info['name']} path distributions ({num_patches} patches)")

    ax2 = axes[1]
    im = ax2.imshow(layer_sums.detach().cpu().numpy().T, aspect='auto', cmap='viridis', vmin=0.0, vmax=1.0)
    ax2.set_xlabel('Patch index')
    ax2.set_ylabel('Depth')
    ax2.set_yticks(np.arange(len(layer_channels)))
    ax2.set_yticklabels([f'L{i+1}' for i in range(len(layer_channels))])
    ax2.set_title('Per-depth path sum (target: 1)')
    plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

    err = (layer_sums - 1.0).abs().max().item()
    print(f"max |depth path-sum - 1|: {err:.3e}")


In [ ]:
def plot_grid(tensor, nrow=8, title=''):
    grid = utils.make_grid(tensor, nrow=nrow, normalize=True, scale_each=True, pad_value=1.0)
    npimg = grid.detach().cpu().numpy()
    plt.figure(figsize=(5, 5))
    plt.title(title)
    plt.axis('off')
    if npimg.shape[0] == 1:
        plt.imshow(npimg[0], cmap='viridis')
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

def visualize_stage_activations(stage_info, max_per_depth=16):
    output = stage_info['output']
    layer_channels = stage_info['layer_channels']
    output_splits = torch.split(output, layer_channels, dim=1)

    for depth_idx, depth_tensor in enumerate(output_splits, start=1):
        fmap = depth_tensor[0].detach().unsqueeze(1)
        n_show = min(max_per_depth, fmap.shape[0])
        plot_grid(
            fmap[:n_show],
            nrow=4,
            title=f"{stage_info['name']} depth {depth_idx} activations",
        )


In [ ]:
# Pick a stage index to inspect (1..4 for ResNet-18 layout).
stage_index = 3
stage_info = details['encoder']['stages'][stage_index - 1]
print(f"Inspecting {stage_info['name']} with depth={stage_info['taxonomy_depth']}")
plot_patch_path_probabilities(stage_info, sample_index=0, max_patches=100)
visualize_stage_activations(stage_info, max_per_depth=16)
